# AI Document Question-Answering System (RAG)

This notebook implements a small Retrieval-Augmented Generation (RAG) system that answers questions using information extracted from a PDF document.

## Project workflow

1. Extract text from the PDF.
2. Split the text into overlapping chunks.
3. Convert chunks into semantic embeddings.
4. Retrieve the most relevant chunks for a question.
5. Generate an answer using a Hugging Face instruction-tuned language model.

## 1. Import libraries

The project uses Sentence Transformers for embeddings, Transformers for answer generation, and PyPDF for PDF text extraction.

In [1]:
import os
import numpy as np
import torch

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

## 2. Load the PDF and extract text

Make sure the PDF is available at the Kaggle input path below. If your dataset has a different folder or filename, update `pdf_path`.

In [2]:
pdf_path = "/kaggle/input/datasets/feyisahabtamu11/document1/document.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(
        f"PDF not found at: {pdf_path}. "
        "Check the Kaggle dataset path and update pdf_path."
    )

reader = PdfReader(pdf_path)

all_text = ""
for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    all_text += page_text + "\n"

print("Pages processed:", len(reader.pages))
print("Total characters extracted:", len(all_text))

if not all_text.strip():
    raise ValueError(
        "No text was extracted. The PDF may contain scanned images"
        "and require OCR."
    )

Pages processed: 6
Total characters extracted: 10082


## 3. Split the document into chunks

Chunking divides the extracted text into smaller sections. The overlap helps preserve context between neighboring chunks.

In [3]:
def chunk_text(text, chunk_size=1000, overlap=200):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be between 0 and chunk_size - 1.")

    paragraphs = [
        paragraph.strip()
        for paragraph in text.split("\n")
        if paragraph.strip()
    ]

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:
        if len(paragraph) > chunk_size:
            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = ""

            start = 0
            while start < len(paragraph):
                end = min(start + chunk_size, len(paragraph))
                chunks.append(paragraph[start:end])
                start += chunk_size - overlap

        elif len(current_chunk) + len(paragraph) + 1 <= chunk_size:
            current_chunk += ("\n" if current_chunk else "") + paragraph

        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = paragraph

    if current_chunk:
        chunks.append(current_chunk)

    return chunks


chunks = chunk_text(all_text)

print("Total chunks:", len(chunks))

Total chunks: 11


## 4. Generate embeddings for the chunks

`all-MiniLM-L6-v2` converts each chunk into a vector representation. Normalized embeddings allow us to compare vectors using the dot product as cosine similarity.

In [4]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (11, 384)


## 5. Load the language model

This project uses Qwen2.5-0.5B-Instruct, a small instruction-tuned causal language model.

In [5]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name)

llm.eval()

print("Language model loaded:", model_name)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Language model loaded: Qwen/Qwen2.5-0.5B-Instruct


## 6. Retrieve relevant chunks

The question is embedded using the same Sentence Transformer model. We calculate similarity against the document embeddings and return the top-k chunks.

In [6]:
def retrieve_relevant_chunks(question, top_k=3):
    query_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarities = embeddings @ query_embedding
    top_indices = np.argsort(similarities)[::-1][:top_k]

    return [chunks[index] for index in top_indices]

## 7. Generate an answer from the retrieved context

The language model receives the retrieved chunks and the question. The prompt instructs it to combine relevant information across chunks, answer every part of the question, and avoid unsupported claims.

`max_length=1024` limits the tokenized input sequence. `max_new_tokens=350` sets the maximum number of tokens generated for the answer.

In [7]:
def generate_answer(question, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are a helpful question-answering assistant.

Answer the question using only the information provided in the context.

Instructions:
1. Read all the context carefully before answering.
2. Combine relevant information from different chunks when necessary.
3. If the question asks for a comparison, explain each concept
   and clearly describe their differences.
4. Answer every part of the question.
5. Keep your answer clear, accurate, and relevant.
6. Do not introduce information that is not supported by the context.
7. If the context does not contain enough information, say:
   "The document does not provide enough information."

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {key: value.to(llm.device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            repetition_penalty=1.1
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

## 8. Ask a question

Change the question below to ask something else about the PDF. The system retrieves relevant chunks and uses them as context for answer generation.

In [8]:
question = (
    "What is the difference between artificial intelligence, "
    "machine learning and deep learning?"
)

retrieved_chunks = retrieve_relevant_chunks(question)
answer = generate_answer(question, retrieved_chunks)

print("Question:")
print(question)

print("\nRetrieved chunks:", len(retrieved_chunks))

print("\nGenerated answer:")
print(answer)

Question:
What is the difference between artificial intelligence, machine learning and deep learning?

Retrieved chunks: 3

Generated answer:
Artificial intelligence (AI), machine learning (ML), and deep learning (DL) are three distinct concepts in the field of artificial intelligence:

1. **Artificial Intelligence (AI)**: It refers to a broad field of computing that focuses on developing intelligent machines capable of performing tasks similar to those performed by humans. Examples include image recognition, natural language processing, predictive analytics, and autonomous vehicles.

2. **Machine Learning (ML)**: A subset of AI that involves algorithms that enable computers to learn from data without being explicitly programmed. ML models are trained on large datasets to recognize patterns and make predictions based on these patterns. Common applications include fraud detection, recommendation systems, and predictive maintenance.

3. **Deep Learning (DL)**: An advanced form of machine

## 9. Project notes

- The system answers from the text extracted from the supplied PDF; it does not search the wider internet.
- Retrieval quality depends on the document content, chunking strategy, embedding model, and selected `top_k`.
- Answer quality also depends on the retrieved context, prompt, language model, and generation settings.
- The evaluation questions and results are documented separately in the project's evaluation report.